# 05 — Model Registry & Versioning

## What
A model registry is a versioned catalogue of trained model artifacts with metadata, lineage, and lifecycle stage. It answers: which model is currently serving production traffic? What were the training data and hyperparameters? What metrics qualified it for promotion?

## Why
Without a registry, teams track production models through Slack messages, spreadsheets, or tribal knowledge. When a model needs to be rolled back — and it will — there is nothing to roll back to. The registry is the single source of truth for which model should be deployed.

## Community context
MLflow Model Registry (open-source) is the community standard for self-hosted stacks. SageMaker Model Registry and Vertex AI Model Registry are the managed equivalents. Both enforce the key workflow: train → register → evaluate → promote → serve → monitor → archive/retrain.

In [ ]:
# Full model registry with lineage tracking
import json, time, hashlib
from pathlib import Path

class ModelVersion:
    VALID_STAGES = ['None', 'Staging', 'Production', 'Archived']

    def __init__(self, name, version, run_id, params, metrics, tags=None):
        self.name = name
        self.version = version
        self.run_id = run_id
        self.params = params
        self.metrics = metrics
        self.tags = tags or {}
        self.stage = 'None'
        self.created_at = time.strftime('%Y-%m-%dT%H:%M:%S')
        self.history = [('None', self.created_at)]

    def transition_to(self, new_stage, user='system'):
        assert new_stage in self.VALID_STAGES, f'Invalid stage: {new_stage}'
        self.stage = new_stage
        self.history.append((new_stage, time.strftime('%Y-%m-%dT%H:%M:%S'), user))

    def __repr__(self):
        return f'ModelVersion({self.name} v{self.version} [{self.stage}])'

class ModelRegistry:
    def __init__(self, store_dir='/tmp/model_registry'):
        self.store = Path(store_dir)
        self.store.mkdir(parents=True, exist_ok=True)
        self._versions = {}  # name -> [ModelVersion]

    def create_registered_model(self, name, description=''):
        self._versions.setdefault(name, [])
        print(f'Registered model: {name}')

    def register_model(self, name, run_id, params, metrics, tags=None):
        self._versions.setdefault(name, [])
        version = len(self._versions[name]) + 1
        mv = ModelVersion(name, version, run_id, params, metrics, tags)
        self._versions[name].append(mv)
        print(f'Registered {name} v{version} from run {run_id[:8]}')
        return mv

    def transition_model_version_stage(self, name, version, new_stage, user='system'):
        mv = self._versions[name][version-1]
        old = mv.stage
        # Auto-archive current Production when promoting new one
        if new_stage == 'Production':
            for v in self._versions[name]:
                if v.stage == 'Production' and v.version != version:
                    v.transition_to('Archived', user)
                    print(f'  Auto-archived {name} v{v.version}')
        mv.transition_to(new_stage, user)
        print(f'Transitioned {name} v{version}: {old} -> {new_stage}')

    def get_latest_versions(self, name, stages=None):
        versions = self._versions.get(name, [])
        if stages:
            versions = [v for v in versions if v.stage in stages]
        return versions

    def search_model_versions(self, name, metric_filter=None):
        versions = self._versions.get(name, [])
        if metric_filter:
            metric, op, val = metric_filter
            versions = [v for v in versions
                        if op == '>' and v.metrics.get(metric, 0) > val]
        return versions

# Demonstrate full promotion workflow
import hashlib
registry = ModelRegistry()
registry.create_registered_model('churn_predictor', 'Predicts customer churn')

# Register two model versions from different training runs
rid1 = hashlib.md5(b'run1').hexdigest()
rid2 = hashlib.md5(b'run2').hexdigest()

mv1 = registry.register_model(
    'churn_predictor', rid1,
    {'lr': 0.01, 'max_depth': 4},
    {'val_auc': 0.82, 'val_recall': 0.71},
    tags={'framework': 'sklearn', 'dataset_version': 'v3'}
)
mv2 = registry.register_model(
    'churn_predictor', rid2,
    {'lr': 0.05, 'max_depth': 6},
    {'val_auc': 0.87, 'val_recall': 0.78},
    tags={'framework': 'sklearn', 'dataset_version': 'v4'}
)

# Standard promotion workflow
print('\n-- Promote v1 to staging --')
registry.transition_model_version_stage('churn_predictor', 1, 'Staging', 'data-scientist')
print('\n-- After validation, promote v2 to production (v1 auto-archived) --')
registry.transition_model_version_stage('churn_predictor', 2, 'Staging', 'ci-system')
registry.transition_model_version_stage('churn_predictor', 2, 'Production', 'ml-platform')

print('\n-- Current Production models --')
for mv in registry.get_latest_versions('churn_predictor', stages=['Production']):
    print(f'  {mv} | auc={mv.metrics["val_auc"]} | run={mv.run_id[:8]}')

print('\n-- All versions with lineage --')
for mv in registry._versions['churn_predictor']:
    print(f'  v{mv.version} [{mv.stage}] auc={mv.metrics["val_auc"]} history={[h[0] for h in mv.history]}')